#  Análise de Performance e Evolução da Cana-de-Açúcar
## Fonte: CONAB — Companhia Nacional de Abastecimento

---

| | |
|---|---|
| 📅 **Safras** | 2017/18 a 2025/26 |
| 📊 **Fonte** | CONAB — Levantamentos Oficiais |
| 🧹 **Notebook** | 01 — Limpeza e Preparação dos Dados |

---

> **Objetivo:** Analisar a evolução da produção, área plantada, qualidade (ATR)
> e derivados (açúcar e etanol) da cana-de-açúcar brasileira ao longo das safras,
> utilizando os levantamentos finais da CONAB por estado (UF).

---

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


## 1. Carregando os dados

In [2]:
df = pd.read_csv('../data/raw/LevantamentoCana.txt', sep=';', encoding='utf-8')
print(f'Shape: {df.shape}')
df.head()

Shape: (742, 14)


,ano_agricola,dsc_safra_previsao,uf,produto,id_produto,dsc_levantamento,id_levantamento,area_plantada_mil_ha,producao_mil_t,producao_acucar_mil_t,producao_etanol_anidro_mil_l,producao_etanol_hidratado_mil_l,producao_etanol_total_mil_l,produtcao_atr_kg_t
0,2017/18,UNICA,AC,CANA DE ACUCAR,4238,1º LEV,1,1.9,104.2,0.0,0.0,6700.0,6700.0,108.7
1,2017/18,UNICA,AC,CANA DE ACUCAR,4238,2º LEV,2,1.9,106.5,0.0,0.0,6463.2,6463.2,102.6
2,2017/18,UNICA,AC,CANA DE ACUCAR,4238,LEVANT,99,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2017/18,UNICA,AL,CANA DE ACUCAR,4238,1º LEV,1,301.7,15538.0,1326.5,246107.0,49624.0,295731.0,123.0
4,2017/18,UNICA,AL,CANA DE ACUCAR,4238,2º LEV,2,325.2,16850.6,1553.2,268717.5,62905.0,331622.4,131.2


## 2. Visão geral do dataset

In [3]:
print('=== TIPOS DE DADOS ===')
print(df.dtypes)
print()
print('=== VALORES NULOS ===')
print(df.isnull().sum())

=== TIPOS DE DADOS ===
ano_agricola                        object
dsc_safra_previsao                  object
uf                                  object
produto                             object
id_produto                           int64
dsc_levantamento                    object
id_levantamento                      int64
area_plantada_mil_ha               float64
producao_mil_t                     float64
producao_acucar_mil_t              float64
producao_etanol_anidro_mil_l       float64
producao_etanol_hidratado_mil_l    float64
producao_etanol_total_mil_l        float64
produtcao_atr_kg_t                 float64
dtype: object

=== VALORES NULOS ===
ano_agricola                        0
dsc_safra_previsao                  0
uf                                  0
produto                             0
id_produto                          0
dsc_levantamento                    0
id_levantamento                     0
area_plantada_mil_ha                0
producao_mil_t                    

## 3. Limpeza — remover espaços extras

In [4]:
# Remover espaços das colunas de texto
df.columns = df.columns.str.strip()
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print('Colunas após limpeza:')
print(df.columns.tolist())

Colunas após limpeza:
['ano_agricola', 'dsc_safra_previsao', 'uf', 'produto', 'id_produto', 'dsc_levantamento', 'id_levantamento', 'area_plantada_mil_ha', 'producao_mil_t', 'producao_acucar_mil_t', 'producao_etanol_anidro_mil_l', 'producao_etanol_hidratado_mil_l', 'producao_etanol_total_mil_l', 'produtcao_atr_kg_t']


## 4. Converter tipos de dados

In [5]:
# Colunas numéricas que podem ter vírgula ou vazio
cols_numericas = [
    'area_plantada_mil_ha', 'producao_mil_t', 'producao_acucar_mil_t',
    'producao_etanol_anidro_mil_l', 'producao_etanol_hidratado_mil_l',
    'producao_etanol_total_mil_l', 'produtcao_atr_kg_t'
]

for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Preencher nulos com 0
df[cols_numericas] = df[cols_numericas].fillna(0)
print('Conversão numérica concluída!')
print(df[cols_numericas].dtypes)

Conversão numérica concluída!
area_plantada_mil_ha               float64
producao_mil_t                     float64
producao_acucar_mil_t              float64
producao_etanol_anidro_mil_l       float64
producao_etanol_hidratado_mil_l    float64
producao_etanol_total_mil_l        float64
produtcao_atr_kg_t                 float64
dtype: object


## 5. Filtrando levantamento final (id = 99)

In [6]:

idx = df.groupby('ano_agricola')['id_levantamento'].transform('max')
df_final = df[df['id_levantamento'] == idx].copy()
df_final = df_final[df_final['producao_mil_t'] > 0]

print(f'Registros levantamento final: {len(df_final)}')
print(f'Safras: {sorted(df_final["ano_agricola"].unique())}')
print(f'Estados: {sorted(df_final["uf"].unique())}')


Registros levantamento final: 177
Safras: ['2017/18', '2018/19', '2019/20', '2020/21', '2021/22', '2022/23', '2023/24', '2024/25', '2025/26']
Estados: ['AL', 'AM', 'BA', 'ES', 'GO', 'MA', 'MG', 'MS', 'MT', 'PA', 'PB', 'PE', 'PI', 'PR', 'RJ', 'RN', 'RO', 'RS', 'SE', 'SP', 'TO']


## 6. Criando colunas derivadas

In [7]:
# Produtividade (ton/ha)
df_final['produtividade_t_ha'] = (
    df_final['producao_mil_t'] / df_final['area_plantada_mil_ha']
).round(2)

# Etanol em mil m³ (mais legível)
df_final['etanol_mil_m3'] = (df_final['producao_etanol_total_mil_l'] / 1000).round(2)

# Região do Brasil
regioes = {
    'SP':'Sudeste','MG':'Sudeste','RJ':'Sudeste','ES':'Sudeste',
    'GO':'Centro-Oeste','MS':'Centro-Oeste','MT':'Centro-Oeste','DF':'Centro-Oeste',
    'PR':'Sul','RS':'Sul','SC':'Sul',
    'AL':'Nordeste','PE':'Nordeste','PB':'Nordeste','RN':'Nordeste',
    'BA':'Nordeste','SE':'Nordeste','MA':'Nordeste','PI':'Nordeste','CE':'Nordeste',
    'PA':'Norte','AM':'Norte','TO':'Norte','RO':'Norte','AC':'Norte','AP':'Norte','RR':'Norte'
}
df_final['regiao'] = df_final['uf'].map(regioes)

print('Colunas criadas com sucesso!')
df_final[['uf','ano_agricola','produtividade_t_ha','etanol_mil_m3','regiao']].head(10)

Colunas criadas com sucesso!


,uf,ano_agricola,produtividade_t_ha,etanol_mil_m3,regiao
5,AL,2017/18,44.92,326.90,Nordeste
8,AM,2017/18,61.69,4.84,Norte
11,BA,2017/18,75.15,180.64,Nordeste
17,ES,2017/18,50.01,90.65,Sudeste
20,GO,2017/18,77.47,4506.50,Centro-Oeste
23,MA,2017/18,58.43,162.66,Nordeste
26,MG,2017/18,78.82,2720.75,Sudeste
29,MS,2017/18,70.48,2632.22,Centro-Oeste
32,MT,2017/18,70.96,1105.62,Centro-Oeste
35,PA,2017/18,72.35,51.56,Norte


## 7. Salvando dados limpos

In [8]:
df_final.to_csv('../data/processed/cana_limpo.csv', index=False, encoding='utf-8')
df.to_csv('../data/processed/cana_todos_levantamentos.csv', index=False, encoding='utf-8')

print('Arquivos salvos:')
print('  - data/processed/cana_limpo.csv (levantamento final)')
print('  - data/processed/cana_todos_levantamentos.csv (todos os levantamentos)')
print(f'\nTotal de registros limpos: {len(df_final)}')

Arquivos salvos:
  - data/processed/cana_limpo.csv (levantamento final)
  - data/processed/cana_todos_levantamentos.csv (todos os levantamentos)

Total de registros limpos: 177
